In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
## reading our closing files which is combined_weekly_dataset
## reading our predicted label regime_label_k
## reading our predicting labels phuc_predictions

## Read core files
regime_df = pd.read_parquet("regime_labeled_k_combine.parquet")
phuc_predictions = pd.read_parquet("test_2layer_labels.parquet")
combined_weekly = pd.read_parquet("combined_weekly_dataset.parquet")

# Make sure indexes are datetime + sorted
regime_df.index = pd.to_datetime(regime_df.index)
phuc_predictions.index = pd.to_datetime(phuc_predictions.index)
combined_weekly.index = pd.to_datetime(combined_weekly.index)

regime_df = regime_df.sort_index()
phuc_predictions = phuc_predictions.sort_index()
combined_weekly = combined_weekly.sort_index()

print("regime_df shape:", regime_df.shape)
print("phuc_predictions shape:", phuc_predictions.shape)
print("combined_weekly shape:", combined_weekly.shape)

print("\nregime_df columns:")
print(regime_df.columns.tolist())

print("\nphuc_predictions columns:")
print(phuc_predictions.columns.tolist())

print("\ncombined_weekly columns:")
print(combined_weekly.columns.tolist())


regime_df shape: (1055, 27)
phuc_predictions shape: (257, 30)
combined_weekly shape: (1631, 31)

regime_df columns:
['global_vol', 'vol_slope', 'vov', 'SP500_SPY_ret', 'NASDAQ100_QQQ_ret', 'DOW_JONES_ret', 'NASDAQ100_ret', 'NIKKEI_225_ret', 'SHANGHAI_COMPOSITE_ret', 'HANG_SENG_ret', 'TREASURY_BOND_TLT_ret', 'GOLD_GLD_ret', 'usd_index_ret', 'SPY_return', 'QQQ_return', 'SPY_vol_12', 'SPY_vol_4', 'SPY_momentum_12', 'VIX', 'VIX_change', 'TREASURY_10Y', 'TREASURY_2Y', 'yield_spread', 'regime', 'dataset_split', 'regime_name', 'regime_3class']

phuc_predictions columns:
['SPY_return', 'QQQ_return', 'SPY_momentum_12', 'VIX_change', 'SPY_vol_4', 'SPY_vol_12', 'yield_spread', 'SPY_return_lag1', 'SPY_return_lag2', 'SPY_return_lag3', 'SPY_return_lag4', 'SPY_return_lag5', 'VIX_change_lag1', 'VIX_change_lag2', 'VIX_change_lag3', 'VIX_change_lag4', 'VIX_change_lag5', 'yield_spread_lag1', 'yield_spread_lag2', 'yield_spread_lag3', 'yield_spread_lag4', 'yield_spread_lag5', 'current_regime', 'next_regime

In [3]:
regime_df.index = pd.to_datetime(regime_df.index, errors="coerce")
phuc_predictions.index = pd.to_datetime(phuc_predictions.index, errors="coerce")
combined_weekly.index = pd.to_datetime(combined_weekly.index, errors="coerce")

regime_df = regime_df[~regime_df.index.isna()].copy()
phuc_predictions = phuc_predictions[~phuc_predictions.index.isna()].copy()
combined_weekly = combined_weekly[~combined_weekly.index.isna()].copy()

regime_df = regime_df[~regime_df.index.duplicated(keep="first")].sort_index()
phuc_predictions = phuc_predictions[~phuc_predictions.index.duplicated(keep="first")].sort_index()
combined_weekly = combined_weekly[~combined_weekly.index.duplicated(keep="first")].sort_index()

print("regime_df date range:", regime_df.index.min(), "to", regime_df.index.max())
print("phuc_predictions date range:", phuc_predictions.index.min(), "to", phuc_predictions.index.max())
print("combined_weekly date range:", combined_weekly.index.min(), "to", combined_weekly.index.max())

regime_df date range: 2004-11-26 00:00:00 to 2026-03-06 00:00:00
phuc_predictions date range: 2021-01-08 00:00:00 to 2026-02-13 00:00:00
combined_weekly date range: 1995-01-06 00:00:00 to 2026-04-03 00:00:00


In [4]:
backtest_df = combined_weekly.copy()

# Keep only useful unsupervised columns
unsup_cols = [col for col in ["regime", "regime_3class", "regime_name"] if col in regime_df.columns]
backtest_df = backtest_df.join(regime_df[unsup_cols], how="left")

# Add Phuc supervised predictions
backtest_df = backtest_df.join(phuc_predictions, how="left")

# Final sort
backtest_df = backtest_df.sort_index()

print("backtest_df shape:", backtest_df.shape)
print("backtest_df date range:", backtest_df.index.min(), "to", backtest_df.index.max())
print("\nbacktest_df columns:")
print(backtest_df.columns.tolist())

backtest_df shape: (1631, 64)
backtest_df date range: 1995-01-06 00:00:00 to 2026-04-03 00:00:00

backtest_df columns:
['SHANGHAI_COMPOSITE', 'CVX', 'usd_index', 'GOLD_GLD', 'REAL_ESTATE_IYR', 'SLB', 'SILVER_SLV', 'CRUDE_OIL_USO', 'MATERIALS_XLB', 'ENERGY_XLE', 'FINANCIALS_XLF', 'INDUSTRIALS_XLI', 'TECH_XLK', 'CONSUMER_STAPLES_XLP', 'UTILITIES_XLU', 'HEALTHCARE_XLV', 'CONSUMER_DISCRETIONARY_XLY', 'XOM', 'DOW_JONES', 'FTSE_100', 'SP500', 'HANG_SENG', 'NIKKEI_225', 'NASDAQ100', 'EURO_STOXX_50', 'VIX', 'TREASURY_10Y', 'TREASURY_2Y', 'NASDAQ100_QQQ', 'SP500_SPY', 'TREASURY_BOND_TLT', 'regime', 'regime_3class', 'regime_name', 'SPY_return', 'QQQ_return', 'SPY_momentum_12', 'VIX_change', 'SPY_vol_4', 'SPY_vol_12', 'yield_spread', 'SPY_return_lag1', 'SPY_return_lag2', 'SPY_return_lag3', 'SPY_return_lag4', 'SPY_return_lag5', 'VIX_change_lag1', 'VIX_change_lag2', 'VIX_change_lag3', 'VIX_change_lag4', 'VIX_change_lag5', 'yield_spread_lag1', 'yield_spread_lag2', 'yield_spread_lag3', 'yield_spread_

In [5]:
# Keep only rows where Phuc's final prediction exists
phuc_bt = backtest_df.dropna(subset=["2l_pred"]).copy()

print("phuc_bt shape:", phuc_bt.shape)
print("phuc_bt date range:", phuc_bt.index.min(), "to", phuc_bt.index.max())

print("\nPhuc signal columns preview:")
print(
    phuc_bt[
        ["current_regime", "next_regime", "target", "actual_next_regime",
         "persist_pred", "2l_base_pred", "2l_pred", "l1_pred"]
    ].head(10)
)

phuc_bt shape: (257, 64)
phuc_bt date range: 2021-01-08 00:00:00 to 2026-02-13 00:00:00

Phuc signal columns preview:
           current_regime next_regime     target actual_next_regime  \
2021-01-08        Low_Vol     Mid_Vol     change            Mid_Vol   
2021-01-15        Mid_Vol     Low_Vol     change            Low_Vol   
2021-01-22        Low_Vol     Mid_Vol     change            Mid_Vol   
2021-01-29        Mid_Vol     Low_Vol     change            Low_Vol   
2021-02-05        Low_Vol     Low_Vol  no change            Low_Vol   
2021-02-12        Low_Vol     Mid_Vol     change            Mid_Vol   
2021-02-19        Mid_Vol     Mid_Vol  no change            Mid_Vol   
2021-02-26        Mid_Vol     Mid_Vol  no change            Mid_Vol   
2021-03-05        Mid_Vol     Low_Vol     change            Low_Vol   
2021-03-12        Low_Vol     Mid_Vol     change            Mid_Vol   

           persist_pred 2l_base_pred  2l_pred    l1_pred  
2021-01-08      Low_Vol      Low_Vol  Low

In [6]:
price_cols = {
    "SPY": "SP500_SPY",
    "QQQ": "NASDAQ100_QQQ",
    "TLT": "TREASURY_BOND_TLT",
    "GLD": "GOLD_GLD",
    "XLE": "ENERGY_XLE",
    "XLK": "TECH_XLK",
    "XLU": "UTILITIES_XLU",
    "XLP": "CONSUMER_STAPLES_XLP",
    "XLV": "HEALTHCARE_XLV",
    "XLF": "FINANCIALS_XLF"
}

returns_df = pd.DataFrame(index=combined_weekly.index)

for asset_name, col in price_cols.items():
    if col in combined_weekly.columns:
        returns_df[f"{asset_name}_return"] = combined_weekly[col].pct_change()
    else:
        print(f"Warning: {col} not found in combined_weekly")

print("Returns df shape:", returns_df.shape)

# Safely overwrite / add return columns into phuc_bt
for col in returns_df.columns:
    phuc_bt[col] = returns_df[col]

print("\nphuc_bt shape after adding returns:", phuc_bt.shape)

print("\nAvailable return columns:")
print([c for c in phuc_bt.columns if c.endswith("_return")])

print("\nReturn preview:")
print(phuc_bt[[c for c in phuc_bt.columns if c.endswith("_return")]].head())

Returns df shape: (1631, 10)

phuc_bt shape after adding returns: (257, 72)

Available return columns:
['SPY_return', 'QQQ_return', 'TLT_return', 'GLD_return', 'XLE_return', 'XLK_return', 'XLU_return', 'XLP_return', 'XLV_return', 'XLF_return']

Return preview:
            SPY_return  QQQ_return  TLT_return  GLD_return  XLE_return  \
2021-01-08    0.019739    0.016861   -0.040639   -0.028145    0.092876   
2021-01-15   -0.014583   -0.022474    0.003304   -0.012749    0.032110   
2021-01-22    0.019111    0.043481    0.000395    0.016186   -0.015906   
2021-01-29   -0.033457   -0.033372    0.000790   -0.007418   -0.065367   
2021-02-05    0.047667    0.053408   -0.024989   -0.016222    0.082401   

            XLK_return  XLU_return  XLP_return  XLV_return  XLF_return  
2021-01-08    0.005691   -0.005901   -0.008154    0.034820    0.048847  
2021-01-15   -0.025543    0.010589   -0.018984   -0.003578    0.000647  
2021-01-22    0.042458   -0.002381   -0.008533    0.005300   -0.019393  
20

In [7]:
print("Returns df shape:", returns_df.shape)

# Safely overwrite / add return columns into phuc_bt
for col in returns_df.columns:
    phuc_bt[col] = returns_df[col]

print("\nphuc_bt shape after adding returns:", phuc_bt.shape)

print("\nAvailable return columns:")
print([c for c in phuc_bt.columns if c.endswith("_return")])

print("\nReturn preview:")
print(phuc_bt[[c for c in phuc_bt.columns if c.endswith("_return")]].head())

print("Phuc backtest shape:", phuc_bt.shape)
print(
    "\nPhuc date range:",
    phuc_bt.index.min().date(), "to", phuc_bt.index.max().date()
)

# Strategy signals from Phuc's model outputs
phuc_bt["signal_persist"] = phuc_bt["persist_pred"]
phuc_bt["signal_2l_base"] = phuc_bt["2l_base_pred"]
phuc_bt["signal_2l"] = phuc_bt["2l_pred"]

print("\nPersistence signal counts:")
print(phuc_bt["signal_persist"].value_counts(dropna=False))

print("\n2-layer base signal counts:")
print(phuc_bt["signal_2l_base"].value_counts(dropna=False))

print("\n2-layer final signal counts:")
print(phuc_bt["signal_2l"].value_counts(dropna=False))

print("\nPreview Phuc signals:")
print(
    phuc_bt[
        [
            "current_regime",
            "actual_next_regime",
            "signal_persist",
            "signal_2l_base",
            "signal_2l"
        ]
    ].head(10)
)

Returns df shape: (1631, 10)

phuc_bt shape after adding returns: (257, 72)

Available return columns:
['SPY_return', 'QQQ_return', 'TLT_return', 'GLD_return', 'XLE_return', 'XLK_return', 'XLU_return', 'XLP_return', 'XLV_return', 'XLF_return']

Return preview:
            SPY_return  QQQ_return  TLT_return  GLD_return  XLE_return  \
2021-01-08    0.019739    0.016861   -0.040639   -0.028145    0.092876   
2021-01-15   -0.014583   -0.022474    0.003304   -0.012749    0.032110   
2021-01-22    0.019111    0.043481    0.000395    0.016186   -0.015906   
2021-01-29   -0.033457   -0.033372    0.000790   -0.007418   -0.065367   
2021-02-05    0.047667    0.053408   -0.024989   -0.016222    0.082401   

            XLK_return  XLU_return  XLP_return  XLV_return  XLF_return  
2021-01-08    0.005691   -0.005901   -0.008154    0.034820    0.048847  
2021-01-15   -0.025543    0.010589   -0.018984   -0.003578    0.000647  
2021-01-22    0.042458   -0.002381   -0.008533    0.005300   -0.019393  
20

In [8]:
# Performance Functions

def cumulative_growth(return_series, initial_value=1.0):
    return_series = return_series.dropna()
    return initial_value * (1 + return_series).cumprod()

def annualized_return(return_series, periods_per_year=52):
    return_series = return_series.dropna()
    n_periods = len(return_series)
    if n_periods == 0:
        return np.nan
    compounded = (1 + return_series).prod()
    return compounded ** (periods_per_year / n_periods) - 1

def annualized_volatility(return_series, periods_per_year=52):
    return_series = return_series.dropna()
    if len(return_series) == 0:
        return np.nan
    return return_series.std() * np.sqrt(periods_per_year)

def sharpe_ratio(return_series, periods_per_year=52, risk_free_rate=0.0):
    return_series = return_series.dropna()
    ann_ret = annualized_return(return_series, periods_per_year)
    ann_vol = annualized_volatility(return_series, periods_per_year)
    if pd.isna(ann_vol) or ann_vol == 0:
        return np.nan
    return (ann_ret - risk_free_rate) / ann_vol

def max_drawdown(return_series):
    return_series = return_series.dropna()
    if len(return_series) == 0:
        return np.nan
    growth = (1 + return_series).cumprod()
    peak = growth.cummax()
    drawdown = growth / peak - 1
    return drawdown.min()

def summarize_performance(return_series, name, periods_per_year=52):
    return {
        "Strategy": name,
        "Annualized Return": annualized_return(return_series, periods_per_year),
        "Annualized Volatility": annualized_volatility(return_series, periods_per_year),
        "Sharpe Ratio": sharpe_ratio(return_series, periods_per_year),
        "Max Drawdown": max_drawdown(return_series)
    }

In [9]:
price_cols = {
    "SPY": "SP500_SPY",
    "QQQ": "NASDAQ100_QQQ",
    "TLT": "TREASURY_BOND_TLT",
    "GLD": "GOLD_GLD",
    "XLE": "ENERGY_XLE",
    "XLK": "TECH_XLK",
    "XLU": "UTILITIES_XLU",
    "XLP": "CONSUMER_STAPLES_XLP",
    "XLV": "HEALTHCARE_XLV",
    "XLF": "FINANCIALS_XLF"
}

allocation_map_a = {
    "Low_Vol": {"SPY": 0.60, "QQQ": 0.20, "TLT": 0.20, "GLD": 0.00},
    "Mid_Vol": {"SPY": 0.30, "QQQ": 0.20, "TLT": 0.30, "GLD": 0.20},
    "High_Vol": {"SPY": 0.10, "QQQ": 0.00, "TLT": 0.70, "GLD": 0.20},
}

allocation_map_b = {
    "Low_Vol": {"SPY": 0.70, "QQQ": 0.20, "TLT": 0.10, "GLD": 0.00},
    "Mid_Vol": {"SPY": 0.50, "QQQ": 0.00, "TLT": 0.30, "GLD": 0.20},
    "High_Vol": {"SPY": 0.10, "QQQ": 0.00, "TLT": 0.70, "GLD": 0.20},
}

allocation_map_c = {
    "Low_Vol": {"SPY": 0.40, "QQQ": 0.20, "XLK": 0.20, "XLE": 0.10, "TLT": 0.10},
    "Mid_Vol": {"TLT": 0.30, "GLD": 0.20, "XLU": 0.35, "XLP": 0.15},
    "High_Vol": {"SPY": 0.10, "QQQ": 0.00, "TLT": 0.70, "GLD": 0.20},
}

allocation_map_d = {
    "Low_Vol": {"SPY": 0.50, "QQQ": 0.20, "XLK": 0.20, "TLT": 0.10},
    "Mid_Vol": {"TLT": 0.10, "GLD": 0.50, "XLU": 0.15, "XLP": 0.10, "XLV": 0.15},
    "High_Vol": {"SPY": 0.10, "QQQ": 0.00, "TLT": 0.70, "GLD": 0.20},
}

In [10]:
# Safely refresh/add all return columns into phuc_bt
for col in returns_df.columns:
    phuc_bt[col] = returns_df[col]

print("phuc_bt shape:", phuc_bt.shape)

print("\nReturn columns available in phuc_bt:")
print([c for c in phuc_bt.columns if c.endswith("_return")])

print("\nPhuc prediction columns preview:")
print(
    phuc_bt[
        [
            "current_regime",
            "actual_next_regime",
            "persist_pred",
            "2l_base_pred",
            "2l_pred",
            "l1_pred"
        ]
    ].head()
)

phuc_bt shape: (257, 75)

Return columns available in phuc_bt:
['SPY_return', 'QQQ_return', 'TLT_return', 'GLD_return', 'XLE_return', 'XLK_return', 'XLU_return', 'XLP_return', 'XLV_return', 'XLF_return']

Phuc prediction columns preview:
           current_regime actual_next_regime persist_pred 2l_base_pred  \
2021-01-08        Low_Vol            Mid_Vol      Low_Vol      Low_Vol   
2021-01-15        Mid_Vol            Low_Vol      Mid_Vol      Mid_Vol   
2021-01-22        Low_Vol            Mid_Vol      Low_Vol      Low_Vol   
2021-01-29        Mid_Vol            Low_Vol      Mid_Vol      Mid_Vol   
2021-02-05        Low_Vol            Low_Vol      Low_Vol      Low_Vol   

            2l_pred    l1_pred  
2021-01-08  Low_Vol  no change  
2021-01-15  Mid_Vol  no change  
2021-01-22  Low_Vol  no change  
2021-01-29  Mid_Vol  no change  
2021-02-05  Low_Vol  no change  


In [11]:
def run_allocation_backtest(
    base_df,
    signal_col,
    allocation_map,
    strategy_name,
    benchmark_spy_col="SPY_return",
    benchmark_tlt_col="TLT_return"
):
    df = base_df.copy()

    # Drop old weight / result columns if rerunning
    cols_to_drop = [c for c in df.columns if c.startswith("w_")] + [
        "strategy_return", "benchmark_spy", "benchmark_80_20",
        "strategy_growth", "spy_growth", "growth_8020"
    ]
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=cols_to_drop)

    def get_weights(regime):
        if pd.isna(regime):
            return {}
        if regime not in allocation_map:
            raise ValueError(f"Regime {regime} not found in allocation map")
        return allocation_map[regime]

    weights_df = df[signal_col].map(get_weights).apply(pd.Series).fillna(0.0)
    weights_df = weights_df.add_prefix("w_")

    df = pd.concat([df, weights_df], axis=1)

    weight_cols = list(weights_df.columns)

    # Ensure needed return columns exist
    for col in weight_cols:
        asset_name = col.replace("w_", "")
        return_col = f"{asset_name}_return"
        if return_col not in df.columns:
            df[return_col] = np.nan
        # do NOT fill missing returns with 0 here anymore

    # Check weight sums
    weight_sums = df[weight_cols].sum(axis=1)
    if not np.allclose(weight_sums.dropna(), 1.0):
        print(f"Warning: some {strategy_name} weights do not sum to 1.0")
        print(weight_sums.value_counts())

    # Old simple logic kept for reference
    '''
    df["strategy_return"] = 0.0
    for col in weight_cols:
        asset_name = col.replace("w_", "")
        df["strategy_return"] += df[col] * df[f"{asset_name}_return"]
    '''

    # New logic: re-scale weights across available assets
    strategy_returns = []

    for idx, row in df.iterrows():
        row_return = 0.0
        valid_weights = {}

        # Keep only assets with non-missing returns
        for col in weight_cols:
            asset_name = col.replace("w_", "")
            return_col = f"{asset_name}_return"

            if return_col in df.columns and pd.notna(row[return_col]):
                valid_weights[asset_name] = row[col]

        # If nothing available, return 0
        if len(valid_weights) == 0:
            strategy_returns.append(0.0)
            continue

        # Re-scale weights to sum to 1 across available assets
        total_weight = sum(valid_weights.values())
        if total_weight == 0:
            strategy_returns.append(0.0)
            continue

        for asset_name, w in valid_weights.items():
            return_col = f"{asset_name}_return"
            row_return += (w / total_weight) * row[return_col]

        strategy_returns.append(row_return)

    df["strategy_return"] = strategy_returns

    # Benchmarks
    df["benchmark_spy"] = df[benchmark_spy_col]
    df["benchmark_80_20"] = 0.80 * df[benchmark_spy_col] + 0.20 * df[benchmark_tlt_col]

    # Growth series
    df["strategy_growth"] = cumulative_growth(df["strategy_return"])
    df["spy_growth"] = cumulative_growth(df["benchmark_spy"])
    df["growth_8020"] = cumulative_growth(df["benchmark_80_20"])

    summary = pd.DataFrame([
        summarize_performance(df["strategy_return"], strategy_name),
        summarize_performance(df["benchmark_spy"], f"Buy & Hold SPY ({strategy_name})"),
        summarize_performance(df["benchmark_80_20"], f"80/20 SPY-TLT ({strategy_name})")
    ])

    return df, summary

In [12]:
results = {}

allocation_tests = {
    "Current": allocation_map_a,
    "Aggressive Bull": allocation_map_b,
    "Sector Tilt": allocation_map_c,
    "Defensive Rotation": allocation_map_d
}

# Create signal columns directly from Phuc's predictions
phuc_bt["signal_persist"] = phuc_bt["persist_pred"]
phuc_bt["signal_2l_base"] = phuc_bt["2l_base_pred"]
phuc_bt["signal_2l"] = phuc_bt["2l_pred"]

for test_name, alloc in allocation_tests.items():
    # Persistence baseline
    persist_res, persist_summary = run_allocation_backtest(
        base_df=phuc_bt.dropna(subset=["signal_persist"]).copy(),
        signal_col="signal_persist",
        allocation_map=alloc,
        strategy_name=f"Persistence - {test_name}"
    )
    persist_summary["Allocation Test"] = test_name
    persist_summary["Model Type"] = "Persistence"

    # Two-layer base model
    base_res, base_summary = run_allocation_backtest(
        base_df=phuc_bt.dropna(subset=["signal_2l_base"]).copy(),
        signal_col="signal_2l_base",
        allocation_map=alloc,
        strategy_name=f"2L Base - {test_name}"
    )
    base_summary["Allocation Test"] = test_name
    base_summary["Model Type"] = "2L Base"

    # Final two-layer model
    two_layer_res, two_layer_summary = run_allocation_backtest(
        base_df=phuc_bt.dropna(subset=["signal_2l"]).copy(),
        signal_col="signal_2l",
        allocation_map=alloc,
        strategy_name=f"2L Final - {test_name}"
    )
    two_layer_summary["Allocation Test"] = test_name
    two_layer_summary["Model Type"] = "2L Final"

    results[test_name] = {
        "persist": (persist_res, persist_summary),
        "base": (base_res, base_summary),
        "final": (two_layer_res, two_layer_summary),
    }

comparison = pd.concat(
    [results[test_name]["persist"][1] for test_name in results] +
    [results[test_name]["base"][1] for test_name in results] +
    [results[test_name]["final"][1] for test_name in results],
    ignore_index=True
)

print(comparison)

                                             Strategy  Annualized Return  \
0                               Persistence - Current           0.092559   
1              Buy & Hold SPY (Persistence - Current)           0.130994   
2               80/20 SPY-TLT (Persistence - Current)           0.092635   
3                       Persistence - Aggressive Bull           0.113666   
4      Buy & Hold SPY (Persistence - Aggressive Bull)           0.130994   
5       80/20 SPY-TLT (Persistence - Aggressive Bull)           0.092635   
6                           Persistence - Sector Tilt           0.149983   
7          Buy & Hold SPY (Persistence - Sector Tilt)           0.130994   
8           80/20 SPY-TLT (Persistence - Sector Tilt)           0.092635   
9                    Persistence - Defensive Rotation           0.140990   
10  Buy & Hold SPY (Persistence - Defensive Rotation)           0.130994   
11   80/20 SPY-TLT (Persistence - Defensive Rotation)           0.092635   
12          

In [13]:
comparison_sorted = comparison.sort_values(
    by=["Model Type", "Sharpe Ratio", "Annualized Return"],
    ascending=[True, False, False]
)

print(comparison_sorted)

                                             Strategy  Annualized Return  \
18                              2L Base - Sector Tilt           0.143782   
21                       2L Base - Defensive Rotation           0.140151   
15                          2L Base - Aggressive Bull           0.109796   
13                 Buy & Hold SPY (2L Base - Current)           0.130994   
16         Buy & Hold SPY (2L Base - Aggressive Bull)           0.130994   
19             Buy & Hold SPY (2L Base - Sector Tilt)           0.130994   
22      Buy & Hold SPY (2L Base - Defensive Rotation)           0.130994   
12                                  2L Base - Current           0.089705   
14                  80/20 SPY-TLT (2L Base - Current)           0.092635   
17          80/20 SPY-TLT (2L Base - Aggressive Bull)           0.092635   
20              80/20 SPY-TLT (2L Base - Sector Tilt)           0.092635   
23       80/20 SPY-TLT (2L Base - Defensive Rotation)           0.092635   
30          

In [14]:
strategy_only = comparison[
    ~comparison["Strategy"].str.contains("Buy & Hold|80/20", regex=True, na=False)
].copy()

print(strategy_only.sort_values(
    by=["Sharpe Ratio", "Annualized Return"],
    ascending=[False, False]
))

                            Strategy  Annualized Return  \
30            2L Final - Sector Tilt           0.168445   
33     2L Final - Defensive Rotation           0.163430   
6          Persistence - Sector Tilt           0.149983   
9   Persistence - Defensive Rotation           0.140990   
18             2L Base - Sector Tilt           0.143782   
27        2L Final - Aggressive Bull           0.131252   
21      2L Base - Defensive Rotation           0.140151   
3      Persistence - Aggressive Bull           0.113666   
24                2L Final - Current           0.107758   
15         2L Base - Aggressive Bull           0.109796   
0              Persistence - Current           0.092559   
12                 2L Base - Current           0.089705   

    Annualized Volatility  Sharpe Ratio  Max Drawdown     Allocation Test  \
30               0.129017      1.305603     -0.198457         Sector Tilt   
33               0.133071      1.228147     -0.221835  Defensive Rotation   
6

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=059a5e93-4459-411b-9a58-2a578fd7a892' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>